# Data collection for Jan Benda

In [1]:
from scipy.io import loadmat, savemat
from scipy.signal import filtfilt, firwin
import numpy as np
import pandas as pd
import sqlite3
import polars as pl
from pathlib import Path

In [3]:
# Data retrieval
pairs = [
    ("ADNFH3", 18) ,
    ("ADNFH4", 18) ,
    ("ADNFH4", 19) ,
    ("ADNFH4", 22) ,
    ("ADNFH4", 23) ,
    ("ADNFH4", 24) ,
    ("ADNFH5", 6 ) ,
    ("ADNFH4", 7 ) ,
    ("ADNFH4", 9 ) ,
    ("ADNFH4", 11) ,
    ("ADNFH4", 12) ,
    ("ADNFH4", 13) ,
    ("ADNFH3", 17) ,
    ("ADNFH4", 17) , #last hd cell
    ("ADNFH4", 27) ,
    ("ADNFH1", 30) ,
    ("ADNFH5", 3 ) ,
    ("ADNFH5", 5 ) ,
    ("ADNFH3", 8 ) ,
    ("ADNFH5", 9 ) ,
    ("ADNFH5", 12) ,
    ("ADNFH5", 15) ,
]

condition = "Baseline"

conn = sqlite3.connect(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\Recordings_FH.db")

placeholders = ",".join(["(?, ?)"] * len(pairs))
query = f"""
    SELECT *
    FROM Recordings
    WHERE (animal_id, cell_id) IN ({placeholders})
"""

params = [x for pair in pairs for x in pair]

df = pd.read_sql_query(query, conn, params=params)

In [21]:
out_path = Path(r"\\172.25.250.112\burgalossi\lab share\Data\Florian\Recordings_for_Benda")

# filter the columns necessary
cols = ["Animal_Id", "Cell_Id", "Folderpath"]
df2 = df[cols]

for r in df2.itertuples():
    animal_id = r.Animal_Id; cell_id = r.Cell_Id; folderpath = r.Folderpath
    datapath = Path(folderpath, 'exp_data.mat')


    loaded = loadmat(datapath, simplify_cells=True)

    # voltage
    voltage = loaded['raw_data']['ephys_data']['traces']
    ephys_sr = loaded['raw_data']['ephys_data']['sampling_rate']

    # tracking
    angles = loaded['processed_data']['tracking_data']['angles']
    tracking_times = loaded['raw_data']['ephys_data']['ttl_times']

    #save to numpy
    np.savez(
        Path(out_path, f"Cell{r.Index+1}.npz"),
        voltage = voltage,
        ephys_sr = ephys_sr,
        angles = angles,
        tracking_times = tracking_times
    )

    #save to mat
    savemat(
        Path(out_path, f"Cell{r.Index+1}.mat"),
        {
            "voltage" : voltage,
            "ephys_sr" : ephys_sr,
            "angles" : angles,
            "tracking_times" : tracking_times
        },
        do_compression=True
        )
